# CNN Tuning Refined: Multimodal EfficientNet Fine-Tuning


In [1]:
# CONFIG

from pathlib import Path

IMAGE_SIZE = (300, 300)
BATCH_SIZE = 16
STAGE1_EPOCHS = 8
STAGE2_EPOCHS = 12
INITIAL_LR = 8e-4
FINE_TUNE_LR = 1e-5
UNFREEZE_LAST_N_LAYERS = 120
SEED = 42
MAX_TOKENS = 120
TEXT_VOCAB_SIZE = 12000
TEXT_EMBED_DIM = 128
META_HIDDEN_DIM = 256
FUSION_HIDDEN_DIM = 512
LABEL_SMOOTHING = 0.1
USE_TTA = True

MODEL_DIR = Path('.')
STAGE1_MODEL_PATH = MODEL_DIR / 'cnn_tuning_refined_stage1.keras'
FINAL_MODEL_PATH = MODEL_DIR / 'cnn_tuning_refined_best.keras'

print({
    'IMAGE_SIZE': IMAGE_SIZE,
    'BATCH_SIZE': BATCH_SIZE,
    'STAGE1_EPOCHS': STAGE1_EPOCHS,
    'STAGE2_EPOCHS': STAGE2_EPOCHS,
    'INITIAL_LR': INITIAL_LR,
    'FINE_TUNE_LR': FINE_TUNE_LR,
    'UNFREEZE_LAST_N_LAYERS': UNFREEZE_LAST_N_LAYERS,
    'MAX_TOKENS': MAX_TOKENS,
    'TEXT_VOCAB_SIZE': TEXT_VOCAB_SIZE,
    'TEXT_EMBED_DIM': TEXT_EMBED_DIM,
    'LABEL_SMOOTHING': LABEL_SMOOTHING,
    'USE_TTA': USE_TTA,
})


{'IMAGE_SIZE': (300, 300), 'BATCH_SIZE': 16, 'STAGE1_EPOCHS': 8, 'STAGE2_EPOCHS': 12, 'INITIAL_LR': 0.0008, 'FINE_TUNE_LR': 1e-05, 'UNFREEZE_LAST_N_LAYERS': 120, 'MAX_TOKENS': 120, 'TEXT_VOCAB_SIZE': 12000, 'TEXT_EMBED_DIM': 128, 'LABEL_SMOOTHING': 0.1, 'USE_TTA': True}


In [2]:
# IMPORTS + DATA LOADING

import ast
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MultiLabelBinarizer
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, callbacks, Model, Input
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import TextVectorization

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

train_df = pd.read_csv('train_split.csv').copy()
val_df = pd.read_csv('val_split.csv').copy()
test_df = pd.read_csv('test_split.csv').copy()

print('train rows:', len(train_df))
print('val rows:', len(val_df))
print('test rows:', len(test_df))


train rows: 35256
val rows: 4407
test rows: 4408


In [3]:
# LABEL MERGING + IMAGE PATH RESOLUTION

def merge_category_phase3(cat):
    if cat in ['tees_tanks', 'graphic_tees', 'shirts_polos', 'blouses_shirts']:
        return 'tops'
    if cat in ['jackets_coats', 'jackets_vests']:
        return 'outerwear'
    if cat in ['sweaters', 'sweatshirts_hoodies', 'cardigans']:
        return 'knitwear'
    if cat in ['shorts', 'skirts', 'pants', 'denim', 'leggings', 'suiting']:
        return 'bottoms'
    if cat in ['rompers / jumpsuits', 'rompers_jumpsuits']:
        return 'rompers_jumpsuits'
    return cat


def resolve_image_path(row):
    masked = row.get('masked_image_path')
    processed = row.get('processed_image_path')
    raw = row.get('image_path')

    for candidate in [masked, processed, raw]:
        if isinstance(candidate, str) and candidate.strip() and Path(candidate).exists():
            return candidate
    return None

for split_df in [train_df, val_df, test_df]:
    if 'category_merged' in split_df.columns:
        split_df['category_ml'] = split_df['category_merged'].apply(merge_category_phase3)
    else:
        split_df['category_ml'] = split_df['category'].apply(merge_category_phase3)
    split_df['model_image_path'] = split_df.apply(resolve_image_path, axis=1)
    split_df['caption_text'] = split_df['caption_text'].fillna('').astype(str)
    split_df = split_df[split_df['model_image_path'].notna()]

train_df = train_df[train_df['model_image_path'].notna()].reset_index(drop=True)
val_df = val_df[val_df['model_image_path'].notna()].reset_index(drop=True)
test_df = test_df[test_df['model_image_path'].notna()].reset_index(drop=True)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['category_ml'])
y_val = label_encoder.transform(val_df['category_ml'])
y_test = label_encoder.transform(test_df['category_ml'])
num_classes = len(label_encoder.classes_)

print('classes:', label_encoder.classes_)
print('num_classes:', num_classes)
print('train usable rows:', len(train_df))
print('val usable rows:', len(val_df))
print('test usable rows:', len(test_df))


classes: ['bottoms' 'dresses' 'knitwear' 'outerwear' 'rompers_jumpsuits' 'tops']
num_classes: 6
train usable rows: 35256
val usable rows: 4407
test usable rows: 4408


In [4]:
# STRUCTURED METADATA / ATTRIBUTE FEATURES
# This gives the deep model access to the same kind of side information that helped Phase 3.

def safe_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(v) for v in parsed]
    except (ValueError, SyntaxError):
        pass
    return []


def attribute_tokens(row):
    tokens = []
    for prefix, column in [('shape', 'shape_labels'), ('fabric', 'fabric_labels'), ('pattern', 'pattern_labels')]:
        values = safe_list(row.get(column))
        for idx, value in enumerate(values):
            tokens.append(f'{prefix}_{idx}_{value}')
    return tokens

meta_cols = ['gender', 'view_name', 'exists_segm', 'exists_caption', 'exists_keypoints']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
train_meta_base = ohe.fit_transform(train_df[meta_cols].fillna('missing').astype(str))
val_meta_base = ohe.transform(val_df[meta_cols].fillna('missing').astype(str))
test_meta_base = ohe.transform(test_df[meta_cols].fillna('missing').astype(str))

mlb = MultiLabelBinarizer()
train_attr = mlb.fit_transform(train_df.apply(attribute_tokens, axis=1).tolist())
val_attr = mlb.transform(val_df.apply(attribute_tokens, axis=1).tolist())
test_attr = mlb.transform(test_df.apply(attribute_tokens, axis=1).tolist())

train_meta = np.concatenate([train_meta_base, train_attr], axis=1).astype('float32')
val_meta = np.concatenate([val_meta_base, val_attr], axis=1).astype('float32')
test_meta = np.concatenate([test_meta_base, test_attr], axis=1).astype('float32')

meta_feature_dim = train_meta.shape[1]
print('meta_feature_dim:', meta_feature_dim)


meta_feature_dim: 103


/opt/anaconda3/envs/swipester/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1007: UserWarning: unknown class(es) ['shape_6_1'] will be ignored
  warnings.warn(


In [5]:
# TEXT VECTORIZATION
# We let the neural model learn from captions directly instead of throwing them away.

text_vectorizer = TextVectorization(
    max_tokens=TEXT_VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_TOKENS,
)
text_vectorizer.adapt(train_df['caption_text'].values)

print('text vocabulary size (actual):', len(text_vectorizer.get_vocabulary()))


text vocabulary size (actual): 106


In [6]:
# TF.DATA PIPELINES

def decode_and_resize(path):
    image_bytes = tf.io.read_file(path)
    image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    return image


def augment_image(image):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.10)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.image.random_saturation(image, lower=0.85, upper=1.15)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image


def build_dataset(df, labels, meta_array, training=False, tta_flip=False):
    paths = df['model_image_path'].tolist()
    captions = df['caption_text'].tolist()

    ds = tf.data.Dataset.from_tensor_slices((paths, captions, meta_array, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED, reshuffle_each_iteration=True)

    def _map_fn(path, caption, meta, label):
        image = decode_and_resize(path)
        if training:
            image = augment_image(image)
        if tta_flip:
            image = tf.image.flip_left_right(image)
        image = preprocess_input(image)
        inputs = {
            'image_input': image,
            'meta_input': tf.cast(meta, tf.float32),
            'caption_input': caption,
        }
        target = tf.one_hot(label, depth=num_classes)
        return inputs, target

    ds = ds.map(_map_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, y_train, train_meta, training=True)
val_ds = build_dataset(val_df, y_val, val_meta, training=False)
test_ds = build_dataset(test_df, y_test, test_meta, training=False)
test_tta_ds = build_dataset(test_df, y_test, test_meta, training=False, tta_flip=True) if USE_TTA else None

class_weights_raw = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
class_weight_dict = {i: float(w) for i, w in enumerate(class_weights_raw)}
print('class_weight_dict:', class_weight_dict)


class_weight_dict: {0: 2.1129090255303846, 1: 1.0759934078007691, 2: 1.4701025769326996, 3: 2.0027266530334016, 4: 4.424698795180723, 5: 0.3133031191682218}


In [7]:
# MODEL DEFINITION
# Image branch: EfficientNetB3
# Text branch: TextVectorization + Embedding + BiGRU
# Meta branch: dense layers over engineered metadata / attribute features
# Fusion: concatenate everything and classify.

def build_multimodal_model(num_classes, meta_feature_dim):
    image_input = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), name='image_input')
    caption_input = Input(shape=(), dtype=tf.string, name='caption_input')
    meta_input = Input(shape=(meta_feature_dim,), dtype=tf.float32, name='meta_input')

    backbone = EfficientNetB3(weights='imagenet', include_top=False, input_tensor=image_input)
    backbone.trainable = False

    image_branch = layers.GlobalAveragePooling2D()(backbone.output)
    image_branch = layers.BatchNormalization()(image_branch)
    image_branch = layers.Dropout(0.35)(image_branch)
    image_branch = layers.Dense(384, activation='relu')(image_branch)

    text_tokens = text_vectorizer(caption_input)
    text_branch = layers.Embedding(TEXT_VOCAB_SIZE, TEXT_EMBED_DIM, mask_zero=True)(text_tokens)
    text_branch = layers.Bidirectional(layers.GRU(64, return_sequences=True))(text_branch)
    text_branch = layers.GlobalMaxPooling1D()(text_branch)
    text_branch = layers.Dropout(0.25)(text_branch)
    text_branch = layers.Dense(128, activation='relu')(text_branch)

    meta_branch = layers.BatchNormalization()(meta_input)
    meta_branch = layers.Dense(META_HIDDEN_DIM, activation='relu')(meta_branch)
    meta_branch = layers.Dropout(0.25)(meta_branch)
    meta_branch = layers.Dense(128, activation='relu')(meta_branch)

    fused = layers.Concatenate()([image_branch, text_branch, meta_branch])
    fused = layers.BatchNormalization()(fused)
    fused = layers.Dropout(0.35)(fused)
    fused = layers.Dense(FUSION_HIDDEN_DIM, activation='relu')(fused)
    fused = layers.Dropout(0.30)(fused)
    output = layers.Dense(num_classes, activation='softmax')(fused)

    model = Model(
        inputs={'image_input': image_input, 'caption_input': caption_input, 'meta_input': meta_input},
        outputs=output,
    )
    return model, backbone

model, backbone = build_multimodal_model(num_classes, meta_feature_dim)
model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR),
    loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy'],
)

model.summary()


43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 78s 2us/step


/opt/anaconda3/envs/swipester/lib/python3.12/site-packages/keras/src/layers/layer.py:1035: UserWarning: Layer 'global_max_pooling1d' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 300, 300,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 300, 300,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 300, 300,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 301, 301,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 150, 150,  │      1,080 │ stem_conv_pad[0]… │
│                     │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 150, 150,  │        160 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 150, 150,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 150, 150,  │        360 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 150, 150,  │        160 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 150, 150,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 40)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 40)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 10)  │        410 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 40)  │        440 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 150, 150,  │          0 │ block1a_activati… │
│ (Multiply)          │ 40)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 150, 150,  │        960 │ block1a_se_excit

 Total params: 13,400,657 (51.12 MB)

 Trainable params: 2,612,564 (9.97 MB)

 Non-trainable params: 10,788,093 (41.15 MB)

In [8]:
# STAGE 1 TRAINING: train the multimodal head with the EfficientNet backbone frozen.
stage1_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=str(STAGE1_MODEL_PATH),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE1_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=stage1_callbacks,
)


Epoch 1/8
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5486 - loss: 1.4222
Epoch 1: val_accuracy improved from None to 0.71296, saving model to cnn_tuning_refined_stage1.keras

Epoch 1: finished saving model to cnn_tuning_refined_stage1.keras
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 7940s 4s/step - accuracy: 0.6122 - loss: 1.2182 - val_accuracy: 0.7130 - val_loss: 1.0080 - learning_rate: 8.0000e-04
Epoch 2/8
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6910 - loss: 1.0096
Epoch 2: val_accuracy improved from 0.71296 to 0.76084, saving model to cnn_tuning_refined_stage1.keras

Epoch 2: finished saving model to cnn_tuning_refined_stage1.keras
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 2914s 1s/step - accuracy: 0.6973 - loss: 0.9956 - val_accuracy: 0.7608 - val_loss: 0.9449 - learning_rate: 8.0000e-04
Epoch 3/8
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7132 - loss: 0.9588
Epoch 3: val_accuracy did not improve from 0.76084
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 3130s 1s/step - accurac

In [9]:
# STAGE 2 FINE-TUNING: unfreeze the top of the full model for a lower-LR pass.
model = tf.keras.models.load_model(STAGE1_MODEL_PATH)

for layer in model.layers:
    layer.trainable = False
for layer in model.layers[-UNFREEZE_LAST_N_LAYERS:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

trainable_count = sum(int(layer.trainable) for layer in model.layers)
print('Trainable layers in stage 2:', trainable_count)

model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy'],
)

fine_tune_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=str(FINAL_MODEL_PATH),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE2_EPOCHS,
    class_weight=class_weight_dict,
    callbacks=fine_tune_callbacks,
)


Trainable layers in stage 2: 97
Epoch 1/12
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7936 - loss: 0.8391
Epoch 1: val_accuracy improved from None to 0.81098, saving model to cnn_tuning_refined_best.keras

Epoch 1: finished saving model to cnn_tuning_refined_best.keras
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 8262s 4s/step - accuracy: 0.7956 - loss: 0.8272 - val_accuracy: 0.8110 - val_loss: 0.8288 - learning_rate: 1.0000e-05
Epoch 2/12
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8078 - loss: 0.7886
Epoch 2: val_accuracy improved from 0.81098 to 0.81280, saving model to cnn_tuning_refined_best.keras

Epoch 2: finished saving model to cnn_tuning_refined_best.keras
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 16839s 8s/step - accuracy: 0.8119 - loss: 0.7804 - val_accuracy: 0.8128 - val_loss: 0.8187 - learning_rate: 1.0000e-05
Epoch 3/12
2204/2204 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.8180 - loss: 0.7624
Epoch 3: val_accuracy improved from 0.81280 to 0.81734, saving model to cnn_t

In [ ]:
# FINAL EVALUATION
# If USE_TTA is True, we average predictions from the original and horizontally flipped test images.

best_model = tf.keras.models.load_model(FINAL_MODEL_PATH if FINAL_MODEL_PATH.exists() else STAGE1_MODEL_PATH)

test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)
print('test_loss:', test_loss)
print('test_accuracy_without_tta:', test_accuracy)

pred_proba = best_model.predict(test_ds, verbose=1)
if USE_TTA:
    pred_proba_tta = best_model.predict(test_tta_ds, verbose=1)
    pred_proba_final = (pred_proba + pred_proba_tta) / 2.0
else:
    pred_proba_final = pred_proba

pred_labels = pred_proba_final.argmax(axis=1)
final_test_accuracy = accuracy_score(y_test, pred_labels)

print('final_test_accuracy:', final_test_accuracy)
print(classification_report(y_test, pred_labels, target_names=label_encoder.classes_))

cm = confusion_matrix(y_test, pred_labels)
cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
display(cm_df)

cnn_tuning_refined_model = best_model
cnn_tuning_refined_label_encoder = label_encoder
cnn_tuning_refined_test_accuracy = float(final_test_accuracy)


/opt/anaconda3/envs/swipester/lib/python3.12/site-packages/keras/src/layers/layer.py:1035: UserWarning: Layer 'global_max_pooling1d' (of type GlobalMaxPooling1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


276/276 ━━━━━━━━━━━━━━━━━━━━ 507s 2s/step - accuracy: 0.8591 - loss: 0.7197
test_loss: 0.7197403311729431
test_accuracy_without_tta: 0.8591197729110718
276/276 ━━━━━━━━━━━━━━━━━━━━ 688s 2s/step
276/276 ━━━━━━━━━━━━━━━━━━━━ 677s 2s/step
final_test_accuracy: 0.8588929219600726
                   precision    recall  f1-score   support

          bottoms       0.69      0.90      0.78       347
          dresses       0.90      0.94      0.92       683
         knitwear       0.73      0.79      0.76       500
        outerwear       0.68      0.89      0.77       367
rompers_jumpsuits       0.81      0.92      0.86       166
             tops       0.96      0.83      0.89      2345

         accuracy                           0.86      4408
        macro avg       0.80      0.88      0.83      4408
     weighted avg       0.88      0.86      0.86      4408



,bottoms,dresses,knitwear,outerwear,rompers_jumpsuits,tops
bottoms,312,0,4,14,1,16
dresses,2,645,3,6,19,8
knitwear,14,0,393,56,0,37
outerwear,8,2,15,327,0,15
rompers_jumpsuits,0,12,0,1,152,1
tops,119,54,125,74,16,1957
